# 실습 7: 학습용과 시험용으로 나누기
- 상황: 아직 검사하지 않은 흐름의 결과를 맞혀보려 한다
- 목표: 답을 아는 기록과 모르는 척할 기록을 나눈다

## Step 0. 정제본 불러오기

In [20]:
import pandas as pd

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

print("행 수, 열 수:", df.shape)
print(df["result"].value_counts())


행 수, 열 수: (1567, 51)
result
양품    1463
불량     104
Name: count, dtype: int64


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 만들 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 지도학습 | 답이 붙어 있는 기록으로 규칙을 찾게 하는 방식. 우리 데이터의 검사 결과가 그 답이다 |
| 분류 | 둘 중 어느 쪽인지 맞히는 문제. 양품이냐 불량이냐 |
| 학습용 | 답을 보여주고 규칙을 찾게 할 몫 |
| 시험용 | 답을 숨겨두고 실력을 재는 데 쓸 몫 |
| 과적합 | 학습용을 통째로 외워버려 처음 보는 기록은 못 맞히는 상태 |
| 일반화 | 그 반대. 처음 보는 기록에도 통하는 상태. 우리가 원하는 것 |
| 클래스 불균형 | 한쪽이 드문 상태. 여기서는 불량이 약 6.6%뿐이다 |
| 층화추출 | 나눌 때 드문 쪽 비율을 양쪽에 똑같이 맞춰주는 방식 |

## Step 2. 빈칸 확인하고 중앙값으로 채우기
센서 열에 빈칸이 몇 개 남아 있는지 확인한 뒤, 그 열의 중앙값으로 채운다. `result` 열은 건드리지 않는다.

In [21]:
# result를 뺀 센서 열 이름만 모은다
sensor_cols = [c for c in df.columns if c != "result"]

빈칸수 = df[sensor_cols].isna().sum()
빈칸있는열 = (빈칸수 > 0).sum()

전체빈칸 = df[sensor_cols].isna().sum().sum()
전체칸 = df[sensor_cols].shape[0] * df[sensor_cols].shape[1]

print("빈칸이 있는 열 수:", 빈칸있는열, "개")
print("전체 빈칸 수:", 전체빈칸, "/", 전체칸)
print("전체 칸 대비 비율:", round(전체빈칸 / 전체칸 * 100, 2), "%")

# 센서 열만 그 열의 중앙값으로 채운다 (result 열은 손대지 않는다)
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

print()
print("채운 뒤 남은 빈칸 수:", df[sensor_cols].isna().sum().sum())


빈칸이 있는 열 수: 48 개
전체 빈칸 수: 1539 / 78350
전체 칸 대비 비율: 1.96 %

채운 뒤 남은 빈칸 수: 0


### 결과 정리 (실행 결과 기준)

| 항목 | 값 |
|---|---|
| 빈칸이 있는 열 수 | 48개 (전체 50개 센서 열 중) |
| 전체 빈칸 수 / 전체 칸 수 | 1,539 / 78,350 |
| 전체 칸 대비 비율 | 1.96% |
| 채운 뒤 남은 빈칸 수 | **0개** |

센서 열 50개 중 48개에 빈칸이 있었고, 전체 칸의 약 2%가 비어 있었다. 각 열의 중앙값으로 채운 뒤에는 빈칸이 하나도 남지 않았다. `df`에 그대로 반영했고, `result` 열은 건드리지 않았다.

In [22]:
# 위 셀에서 채우기 전에 구해둔 '빈칸수'를 그대로 써서, 빈칸이 많은 순으로 5개만 본다
빈칸수.sort_values(ascending=False).head(5)


sensor_248    715
sensor_552    260
sensor_551    260
sensor_091     51
sensor_080     24
dtype: int64

### 결과 정리 (실행 결과 기준)

| 순위 | 센서 | 빈칸 수 |
|---|---|---|
| 1 | sensor_248 | 715 |
| 2 | sensor_552 | 260 |
| 3 | sensor_551 | 260 |
| 4 | sensor_091 | 51 |
| 5 | sensor_080 | 24 |

## Step 3. 정답표를 숫자로 바꾸기
`result` 열(양품/불량)을 그대로 두고, 숫자로 바꾼 `불량여부` 열(불량=1, 양품=0)을 새로 만든다.

In [23]:
# result는 그대로 두고, 숫자로 바꾼 답 열을 새로 만든다 (불량=1, 양품=0)
df["불량여부"] = (df["result"] == "불량").astype(int)

print(df[["result", "불량여부"]].head())


  result  불량여부
0     양품     0
1     양품     0
2     불량     1
3     양품     0
4     양품     0


## Step 4. 입력과 정답으로 가르기
센서 열은 입력(X), 방금 만든 `불량여부`는 정답(y)으로 나눈다.

In [24]:
# 센서 열은 입력(X), 불량여부는 정답(y)
X = df[sensor_cols]
y = df["불량여부"]

print("X 행 수, 열 수:", X.shape)
print("y 행 수:", y.shape)


X 행 수, 열 수: (1567, 50)
y 행 수: (1567,)


## Step 5. 학습용과 시험용으로 나누기
불량 비율이 낮으니, 층화추출(stratify)로 양쪽 몫에 불량 비율이 비슷하게 나오도록 나눈다.

In [25]:
from sklearn.model_selection import train_test_split

# stratify=y — 나눌 때 양품/불량 비율을 양쪽에 똑같이 맞춘다
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("y_train:", y_train.shape, " y_test:", y_test.shape)


X_train: (1253, 50)  X_test: (314, 50)
y_train: (1253,)  y_test: (314,)


### 문법 노트 - 나누기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| train_test_split(X, y) | 입력과 정답을 같은 기준으로 두 몫씩 갈라준다 | 답을 숨겨둔 몫이 있어야 실력을 잰다 |
| test_size=0.2 | 시험용으로 뗄 비율 | 20%면 300건 정도 남는다 |
| random_state=42 | 섞는 방식을 고정 | 안 넣으면 돌릴 때마다 결과가 달라져 비교가 안 된다 |
| stratify=y | 정답 비율을 양쪽에 맞춰 나눈다 | 불량이 6.6%뿐이라 안 맞추면 한쪽에 몰린다 |

**돌려주는 것이 네 덩어리인 순서에 주의.**<br>
X_train, X_test, y_train, y_test 순서다. 입력 둘이 먼저, 정답 둘이 나중.<br>
순서를 바꿔 받으면 오류 없이 실행되면서 결과만 이상해진다.

## Step 6. 제대로 나뉘었는지 확인하기
X_train, X_test, y_train, y_test 각각의 행 수와 불량 비율을 원본 전체와 비교한다.

In [26]:
행목록 = []
for 이름, y부분 in [("X_train / y_train", y_train), ("X_test / y_test", y_test)]:
    행수 = len(y부분)
    불량건수 = (y부분 == 1).sum()
    불량비율 = round(불량건수 / 행수 * 100, 2)
    행목록.append({"구분": 이름, "행 수": 행수, "불량 건수": 불량건수, "불량 비율(%)": 불량비율})

전체불량비율 = round(y.mean() * 100, 2)

요약표 = pd.DataFrame(행목록)
요약표["원본 전체 불량 비율(%)"] = 전체불량비율
요약표


,구분,행 수,불량 건수,불량 비율(%),원본 전체 불량 비율(%)
0,X_train / y_train,1253,83,6.62,6.64
1,X_test / y_test,314,21,6.69,6.64


[나눈 결과]<br>
학습용 : [1253]건 (불량 [83]건, [6.62]%)<br>
시험용 : [314]건 (불량 [21]건, [6.69]%)<br>
전체   : [1567]건 (불량 [104]건, [6.64]%)<br>
시험용 불량이 [21]건뿐이다.

### 결과 정리 (실행 결과 기준)

| 구분 | 행 수 | 불량 건수 | 불량 비율(%) |
|---|---|---|---|
| X_train / y_train | 1253 | 83 | 6.62 |
| X_test / y_test | 314 | 21 | 6.69 |

원본 전체 불량 비율: **6.64%**

층화추출 덕분에 학습용(6.62%)과 시험용(6.69%) 모두 원본(6.64%)과 거의 같은 불량 비율을 유지했다.

## Step 7. 첫 예측 한 번 돌려보기
가장 기본적인 분류 모델(의사결정나무)을 X_train, y_train으로 학습시키고, X_test에 대해 예측만 해본다. 점수는 아직 계산하지 않는다.

In [27]:
from sklearn.tree import DecisionTreeClassifier

# 분류에서 가장 기본이 되는 모델 중 하나 — 의사결정나무
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

예측 = model.predict(X_test)

print("1) 예측 결과 개수:", len(예측), "개")
print("2) 불량이라고 예측한 건수:", (예측 == 1).sum(), "건")
print("3) 시험용의 실제 불량:", (y_test == 1).sum(), "건")

print()
print("[예측한 값의 종류별 개수]")
라벨 = pd.Series(예측).map({0: "양품(0)", 1: "불량(1)"})
print(라벨.value_counts())


1) 예측 결과 개수: 314 개
2) 불량이라고 예측한 건수: 31 건
3) 시험용의 실제 불량: 21 건

[예측한 값의 종류별 개수]
양품(0)    283
불량(1)     31
Name: count, dtype: int64


### 결과 정리 (실행 결과 기준)

1. 예측 결과 개수: **314개**
2. 불량이라고 예측한 건수: **31건**
3. 시험용의 실제 불량 건수: **21건**

[예측한 값의 종류별 개수]

| 값 | 개수 |
|---|---|
| 양품(0) | 283 |
| 불량(1) | 31 |

의사결정나무는 로지스틱 회귀와 달리 값의 크기(스케일)에 영향을 받지 않아서, 불량을 아예 안 찍는 문제 없이 31건을 불량으로 예측했다. 실제 불량(21건)보다 많은 31건을 예측했다는 것은 맞힌 것도 있고 잘못 짚은 것도 섞여 있다는 뜻인데, 정확도 등 점수는 다음 단계에서 계산한다.

> ⚠️ **불량이라고 예측한 건수는 사람마다 크게 다릅니다.** 0건이 나와도, 서른 건 넘게 나와도 정상이에요. 어떤 모델을 추천받았느냐에 따라 갈립니다.
> 
> 
> **불량 예측 0건** — 모델이 "전부 양품"이라고 답해버린 것입니다. 불량이 워낙 드물어서 그게 제일 안전하다고 배웠어요. 그러면 정확도는 93점쯤 나옵니다 — 하나도 못 잡았는데요.
> 
> **불량 예측 20~35건** — 모델이 꽤 적극적으로 지목한 것입니다. 실제 불량은 21건인데 그보다 많이 지목했다면 그중 상당수가 헛짚은 거예요. 이 경우 정확도는 오히려 **84~89%로 떨어집니다.**
> 
> **두 번째 경우가 특히 재미있습니다.** 정확도로 보면 아무것도 안 하는 것보다 못한데, 불량은 더 잡았거든요. 어느 쪽이 나은 걸까요?
> 
> **그 판단을 할 자가 아직 우리에게 없습니다.** 다음 두 실습에서 그 자를 만듭니다. 지금은 "돌아가긴 한다, 그런데 점수 하나로는 뭐라 말 못 하겠다"까지만 보고 넘어가세요.
> 
> 📌 **본인 숫자를 적어두세요.** 다음 실습에서 이 숫자와 비교합니다.
>

## 도전 - stratify 유무 비교
random_state를 0~4로 바꿔가며, stratify=y를 넣은 경우와 뺀 경우를 나란히 비교한다. 앞에서 만든 X_train 등은 건드리지 않는다.

In [ ]:
비교결과 = []
for rs in range(5):
    # stratify 없이 나눈 경우
    _, X테스트_없음, _, y테스트_없음 = train_test_split(X, y, test_size=0.2, random_state=rs)
    없음_불량건수 = (y테스트_없음 == 1).sum()
    없음_불량비율 = round(없음_불량건수 / len(y테스트_없음) * 100, 2)

    # stratify=y로 나눈 경우
    _, X테스트_있음, _, y테스트_있음 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    있음_불량건수 = (y테스트_있음 == 1).sum()
    있음_불량비율 = round(있음_불량건수 / len(y테스트_있음) * 100, 2)

    비교결과.append({
        "random_state": rs,
        "stratify 없음 - 불량 건수": 없음_불량건수,
        "stratify 없음 - 불량 비율(%)": 없음_불량비율,
        "stratify=y - 불량 건수": 있음_불량건수,
        "stratify=y - 불량 비율(%)": 있음_불량비율,
    })

비교표 = pd.DataFrame(비교결과)
비교표


### 결과 정리 (실행 결과 기준)

| random_state | stratify 없음 - 불량 건수 | stratify 없음 - 불량 비율(%) | stratify=y - 불량 건수 | stratify=y - 불량 비율(%) |
|---|---|---|---|---|
| 0 | 13 | 4.14 | 21 | 6.69 |
| 1 | 20 | 6.37 | 21 | 6.69 |
| 2 | 20 | 6.37 | 21 | 6.69 |
| 3 | 19 | 6.05 | 21 | 6.69 |
| 4 | 26 | 8.28 | 21 | 6.69 |

stratify 없이 나누면 random_state에 따라 시험용 불량 건수가 13~26건까지 들쭉날쭉하고 비율도 4.14%~8.28%로 흔들린다. stratify=y를 넣으면 random_state를 뭘로 바꾸든 항상 21건(6.69%)으로 원본(6.64%)에 가깝게 고정된다.